# Deadtime explorer
Interactive filters to explore the deadtime data without rendering every plot.

In [1]:

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Checkbox
import sys

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / 'src'))
from deadtime_analysis import DeadtimeAnalysis

EST_PATH = ROOT / 'data' / 'estimated_deadtime_all.json'
COMBINED = ROOT / 'data' / 'combined_deadtime.jsonl'
#COMBINED = ROOT / 'data' / 'double_pulse_deadtime-11-19-25.jsonl'

# Load estimates
with EST_PATH.open() as fh:
    estimates = json.load(fh)

est_df = pd.DataFrame(estimates)

# Load full data
analysis_full = DeadtimeAnalysis.from_jsonl([str(COMBINED)])

print('Estimates rows:', len(est_df))
print('Data rows:', len(analysis_full.df))
print('Available pulse rates:', sorted(analysis_full.df['pulse_rate_hz'].dropna().unique()))


Estimates rows: 252
Data rows: 4033
Available pulse rates: [np.float64(10.0), np.float64(100.0)]


## Interactive slice
Select pulse rate, pulse count, channels, and windows to explore the data.

In [2]:

# Build widget options from all data
pulse_rate_options = sorted(analysis_full.df['pulse_rate_hz'].dropna().unique())
pulse_options = sorted(analysis_full.df['num_pulses'].dropna().unique())
channel_options = ['(all)'] + sorted(analysis_full.df['channel_count'].dropna().unique())
window_options = ['(all)'] + sorted(analysis_full.df['windows'].dropna().unique())

# Y-axis bounds - set to None for auto, or set numeric values
# Example: y_min = 0, y_max = 1000
y_min = None  # Set to None for auto, or a number like 0
y_max = 22.5  # Set to None for auto, or a number like 1000

@interact(
    pulse_rate=Dropdown(options=pulse_rate_options, value=pulse_rate_options[0] if pulse_rate_options else None, description='Pulse Rate (Hz)'),
    num_pulses=Dropdown(options=pulse_options, value=pulse_options[0] if pulse_options else None, description='Pulses'),
    channels=Dropdown(options=channel_options, value='(all)', description='Channels'),
    windows=Dropdown(options=window_options, value='(all)', description='Windows'),
    show_stars=Checkbox(value=True, description='Show stars'),
    show_vertical_bounds=Checkbox(value=True, description='Show vertical bounds'),
    show_notes=Checkbox(value=True, description='Show notes'),
)
def _plot(pulse_rate, num_pulses, channels, windows, show_stars, show_vertical_bounds, show_notes):
    if pulse_rate is None or num_pulses is None:
        print('No data available')
        return
    
    # Filter data by pulse rate
    df = analysis_full.df[analysis_full.df['pulse_rate_hz'] == pulse_rate].copy()
    df = df[df['num_pulses'] == num_pulses]
    
    ch_val = None if channels == '(all)' else channels
    win_val = None if windows == '(all)' else windows
    if ch_val is not None:
        df = df[df['channel_count'] == ch_val]
    if win_val is not None:
        df = df[df['windows'] == win_val]
    if df.empty:
        print('No data for selection')
        return
    
    # Build estimate match for highlights
    match = None
    est_match = est_df[(est_df['pulse_rate_hz'] == pulse_rate) & (est_df['num_pulses'] == num_pulses)]
    if ch_val is not None:
        est_match = est_match[est_match['channel_count'] == ch_val]
    if win_val is not None:
        est_match = est_match[est_match['windows'] == win_val]
    if not est_match.empty:
        match = est_match.iloc[0]
    
    highlights = None
    deadtime_range_ns = None
    deadtime_range_text = None
    if match is not None:
        lb = match.get('min_all_pulses_lower_bound_ns')
        resp = match.get('min_all_pulses_response_ns')
        if lb is not None and resp is not None:
            highlights = [lb, resp]
            deadtime_range_ns = (lb, resp)
            deadtime_range_text = f"Deadtime range: {lb:.0f} - {resp:.0f} ns"
    
    ana = DeadtimeAnalysis(df, single_factor=analysis_full.single_factor, double_factor=analysis_full.double_factor)
    ana.plot_rate_vs_separation_by_channels(
        pulse_rate, 
        num_pulses=num_pulses,
        highlight_separations_ns=highlights,
        show_stars=show_stars,
        show_vertical_bounds=show_vertical_bounds,
        show_notes=show_notes,
        deadtime_range_ns=deadtime_range_ns,
        deadtime_range_text=deadtime_range_text,
        y_min=y_min,
        y_max=y_max,
    )
    ana.plot_rate_vs_separation_by_windows(
        pulse_rate, 
        num_pulses=num_pulses,
        highlight_separations_ns=highlights,
        show_stars=show_stars,
        show_vertical_bounds=show_vertical_bounds,
        show_notes=show_notes,
        deadtime_range_ns=deadtime_range_ns,
        deadtime_range_text=deadtime_range_text,
        y_min=y_min,
        y_max=y_max,
    )


interactive(children=(Dropdown(description='Pulse Rate (Hz)', options=(np.float64(10.0), np.float64(100.0)), v…